## Imports

In [1]:
import os
import sys
sys.path.insert(0, os.path.abspath('/home/ec2-user/multimodal-rag'))

import nest_asyncio
nest_asyncio.apply()

import pandas as pd
import json
from tqdm.notebook import tqdm
from scripts.vlm import VLM
from scripts.ocr import OCRRetriever
from scripts.evaluation import evaluate_dataframe

## Configurations

In [2]:
SEED = 19
QUESTION_PATHS = {
    "FinancialReport" : "../data/raw/REAL-MM-RAG_FinReport/test/qas.jsonl",
    "TechReport" : "../data/raw/REAL-MM-RAG_TechReport/test/qas.jsonl",
    "TechSlides" : "../data/raw/REAL-MM-RAG_TechSlides/test/qas.jsonl"
}
RESULTS_PATH = "./results/ocr_rag"
model_id = "Qwen/Qwen3-VL-8B-Instruct"
checkpoint_path = "../Qwen3-VL-8B-Instruct"
OCR_EMBEDDINGS_DIR = "../scripts/embeddings_ocr"
TOP_K = 3
RETRIEVAL_FILES = {
    "FinancialReport": f"{RESULTS_PATH}/retrieval_finreport.json",
    "TechReport":      f"{RESULTS_PATH}/retrieval_techreport.json",
    "TechSlides":      f"{RESULTS_PATH}/retrieval_techslides.json",
}

## Load Data

Reproduce the identical 80/20 train/test split used in the baseline.

In [3]:
dataframes = {}

for key, path in QUESTION_PATHS.items():
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            records.append({
                "example_index": obj.get("example_index"),
                "question": obj.get("question"),
                "answer": obj.get("answer"),
                "intended_img": obj.get("image_path")
            })
    dataframes[key] = pd.DataFrame(records).set_index("example_index")

splits = {}
for key, df in dataframes.items():
    train = df.sample(frac=0.8, random_state=SEED)
    test = df.drop(train.index)
    splits[key] = {"train": train, "test": test}
    print(f"{key}: {len(train)} train / {len(test)} test")

FinancialReport: 682 train / 171 test
TechReport: 1035 train / 259 test
TechSlides: 1083 train / 271 test


## Retrieval Phase

Run the cells in this section to retrieve top-k page images per query using ColQwen and save image paths to JSON. Then **restart the kernel** before running the VLM Inference section — ColQwen and QwenVL-8B cannot coexist in GPU memory.

In [4]:
retriever = OCRRetriever(persist_dir=OCR_EMBEDDINGS_DIR)

os.makedirs(RESULTS_PATH, exist_ok=True)
for name, qas_path in QUESTION_PATHS.items():
    print(f"Ingesting {name}...")
    retriever.ingest_dataset(qas_path)
print("\n✅ All datasets ingested.")

for key, split in splits.items():
    test_df = split['test']
    retrieval_path = RETRIEVAL_FILES[key]

    print(f"Retrieving for {key}...")
    records = []
    for idx, row in tqdm(test_df.iterrows(), total=len(test_df)):
        results = retriever.search(row['question'], top_k=TOP_K)
        context_parts = []
        for i, (_rid, _sim, meta) in enumerate(results):
            text_path = meta.get("text_path", "")
            if text_path and os.path.exists(text_path):
                try:
                    with open(text_path, "r", encoding="utf-8", errors="ignore") as fh:
                        text = fh.read().strip()
                    if text:
                        context_parts.append(f"[Document {i + 1}]\n{text}")
                except Exception as e:
                    print(f"  WARNING: could not read {text_path}: {e}")

        if context_parts:
            context = "\n\n".join(context_parts)
            prompt = f"Context:\n{context}\n\nQuestion: {row['question']}"
        else:
            prompt = row['question']

        records.append({
            "example_index": idx,
            "dataset": key,
            "question": row['question'],
            "reference": row['answer'],
            "retrieved_prompt": prompt,
        })

    with open(retrieval_path, "w") as f:
        json.dump(records, f, indent=2)
    print(f"  Saved {len(records)} records → {retrieval_path}")

print("\n✅ Retrieval complete.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Ingesting FinancialReport...


Indexing OCR pages: 100%|██████████| 853/853 [00:03<00:00, 273.65it/s] 


Ingesting TechReport...


...

  Saved 171 records → ./results/ocr_rag/retrieval_finreport.json
Retrieving for TechReport...


  0%|          | 0/259 [00:00<?, ?it/s]

  Saved 259 records → ./results/ocr_rag/retrieval_techreport.json
Retrieving for TechSlides...


  0%|          | 0/271 [00:00<?, ?it/s]

  Saved 271 records → ./results/ocr_rag/retrieval_techslides.json

✅ Retrieval complete.


In [4]:
# Build a lookup: {(dataset_key, example_index) -> retrieved_prompt}
retrieval_lookup = {}
for key, path in RETRIEVAL_FILES.items():
    with open(path, "r", encoding="utf-8") as f:
        for rec in json.load(f):
            retrieval_lookup[(key, rec["example_index"])] = rec["retrieved_prompt"]

print(f"Loaded {len(retrieval_lookup)} retrieval records.")

Loaded 701 retrieval records.


## VLM Inference

After restarting the kernel, re-run Imports, Configurations, and Load Data, then run the cells below.

In [4]:
vlm = VLM(model_id, checkpoint_path)

--- Local model found at: ../Qwen3-VL-8B-Instruct ---
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0


Multi-thread loading shards: 100% Completed | 4/4 [02:10<00:00, 32.70s/it]
Capturing batches (bs=24 avail_mem=1.75 GB):   0%|          | 0/7 [00:00<?, ?it/s]2026-04-24 02:14:45,002 - CUTE_DSL - WARNING - [handle_import_error] - Unexpected error during package walk: cutlass.cute.experimental
[2026-04-24 02:14:45] Unexpected error during package walk: cutlass.cute.experimental
Capturing batches (bs=1 avail_mem=1.68 GB): 100%|██████████| 7/7 [00:06<00:00,  1.14it/s] 


In [7]:
os.makedirs(RESULTS_PATH, exist_ok=True)
output_filepath = f"{RESULTS_PATH}/ocr_rag_results.jsonl"

with open(output_filepath, "w", encoding="utf-8") as f:
    for key, split in splits.items():
        test_df = split['test']
        print(f"Running Inference for {key}")

        for idx, row in tqdm(test_df.iterrows(), total=len(test_df)):
            retrieved_prompt = retrieval_lookup.get((key, idx))
            if retrieved_prompt is None:
                print(f"  WARNING: no retrieval for {key} example {idx}, skipping")
                continue

            messages = [
                {
                    "role": "system",
                    "content": [{"type": "text", "text": "You are a question answering assistant for corporate applications. Respond in one sentence using all available information."}]
                },
                {
                    "role": "user",
                    "content": [{"type": "text", "text": retrieved_prompt}]
                }
            ]

            response = vlm.generate(messages)

            result_obj = {
                "dataset": key,
                "example_index": idx,
                "question": row['question'],
                "generated_answer": response['text'],
                "ground_truth": row['answer'],
                "intended_img": row['intended_img']
            }

            f.write(json.dumps(result_obj) + "\n")
            f.flush()

print(f"✅ Inference complete. Results saved to {output_filepath}")

Running Inference for FinancialReport


  0%|          | 0/171 [00:00<?, ?it/s]

Running Inference for TechReport


  0%|          | 0/259 [00:00<?, ?it/s]

Running Inference for TechSlides


  0%|          | 0/271 [00:00<?, ?it/s]

✅ Inference complete. Results saved to ./results/ocr_rag/ocr_rag_results.jsonl


## Evaluation

In [5]:
input_filepath = f"{RESULTS_PATH}/ocr_rag_results.jsonl"
report_filepath = f"{RESULTS_PATH}/ocr_rag_evaluation_report.txt"
detailed_jsonl_filepath = f"{RESULTS_PATH}/ocr_rag_evaluated_details.jsonl"

print(f"Loading data from {input_filepath}...")
records = []
if os.path.exists(input_filepath):
    with open(input_filepath, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
else:
    raise FileNotFoundError(f"Could not find {input_filepath}. Check your paths!")

df_results = pd.DataFrame(records)
print(f"Loaded {len(df_results)} records across datasets: {df_results['dataset'].unique()}")
print("\nStarting evaluation (this may take a few minutes depending on BERTScore)...")
evaluated_df = evaluate_dataframe(df_results, report_filepath)

evaluated_df.to_json(detailed_jsonl_filepath, orient="records", lines=True)
print(f"✅ Detailed row-by-row scores saved to {detailed_jsonl_filepath}")

Loading data from ./results/ocr_rag/ocr_rag_results.jsonl...
Loaded 701 records across datasets: ['FinancialReport' 'TechReport' 'TechSlides']

Starting evaluation (this may take a few minutes depending on BERTScore)...
Calculating BLEU scores...
Calculating ROUGE scores...


Calculating BERT scores (this may take a moment)...


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Report successfully saved to ./results/ocr_rag/ocr_rag_evaluation_report.txt
✅ Detailed row-by-row scores saved to ./results/ocr_rag/ocr_rag_evaluated_details.jsonl
